In [15]:
import joblib
import geopandas as gpd
import numpy as np
import pandas as pd
from pathlib import Path

BASE_DIR = Path(r"C:\Users\anils\Desktop\7th SEM\GIS\GeoOptim\Output")

# ==============================================================================
# Load Trained Artifacts and Dataset
# ==============================================================================
print("⏳ Loading model pipeline artifacts...")
scaler = joblib.load(BASE_DIR/"Models/robust_scaler.pkl")
model = joblib.load(BASE_DIR/"Models/mlp_multilabel_model.pkl")

hex_gdf = gpd.read_file(r"C:\Users\anils\Desktop\7th SEM\GIS\GeoOptim\Data\Final\hexagon_ktm_V2.geojson")

# ==============================================================================
# 3. Dynamic Feature Column Extraction
# ==============================================================================
# Define the non-feature tracking attributes to isolate your dynamic feature matrix
metadata_cols = ['h3_id', 'geometry']
feature_cols = [col for col in hex_gdf.columns if col not in metadata_cols and not col.startswith('prob_')]

# The target multi-label classes your network yields output arrays for
clean_label_names = [
    'Restaurant', 'Hotel_Lodging', 'Malls_Department', 'Cafe_Bakery' 
    , 'Fashion_Clothing', 'Convenience_Specialty'
]

X_inference = hex_gdf[feature_cols].copy()

⏳ Loading model pipeline artifacts...


In [16]:
# Ensure feature counts align with what the scaler expects
expected_features = scaler.n_features_in_
actual_features = len(feature_cols)
if actual_features != expected_features:
    raise ValueError(f"❌ Feature count mismatch! Model expects {expected_features}, but dataset has {actual_features}.")

# ==============================================================================
# 4. Run Scaling and Predict Probabilities (FIXED FOR MLP)
# ==============================================================================
print("⚖️ Normalizing features using loaded RobustScaler...")
X_inference_scaled = scaler.transform(X_inference)

⚖️ Normalizing features using loaded RobustScaler...


In [17]:
print("🧠 Running Multi-Label MLP model inference engine...")
# ✨ FIX: For MLP Multi-label, y_prob is a single 2D array of shape (n_samples, n_classes)
y_prob_matrix = model.predict_proba(X_inference_scaled)

print("📈 Appending class probabilities to dataset...")
for idx, label_name in enumerate(clean_label_names):
    # Slice the matrix column directly for each target class
    hex_gdf[f'prob_{label_name}'] = y_prob_matrix[:, idx].astype('float32')

# ==============================================================================
# 5. Export Updated Geographic Layer File
# ==============================================================================
print(f"💾 Saving complete prediction probability field matrix...")
hex_gdf.to_file(BASE_DIR/"Plots/mlp_output.geojson", driver="GeoJSON")

print("\n🎉 Model Inference Run Complete!")
print(f"Saved completed predictions with {len(clean_label_names)} probability fields")

🧠 Running Multi-Label MLP model inference engine...
📈 Appending class probabilities to dataset...
💾 Saving complete prediction probability field matrix...

🎉 Model Inference Run Complete!
Saved completed predictions with 6 probability fields


In [14]:
y_prob_matrix.shape

(6, 33013, 2)

In [18]:
import geopandas as gpd
df = gpd.read_file(r"C:\Users\anils\Desktop\7th SEM\GIS\GeoOptim\Output\Plots\mlp_output.geojson")

df.head()

,h3_id,Education_Hubs_500m,Education_Hubs_1km,Education_Hubs_2km,Social_Community_Services_500m,Social_Community_Services_1km,Social_Community_Services_2km,Restaurant_500m,Restaurant_1km,Restaurant_2km,...,elevation_mean,elevation_std,slope_mean_deg,prob_Restaurant,prob_Hotel_Lodging,prob_Malls_Department,prob_Cafe_Bakery,prob_Fashion_Clothing,prob_Convenience_Specialty,geometry
0,8a3c02d4105ffff,9,15,19,1,2,6,1,3,7,...,1429.945801,2.065357,3.188434,0.384930,0.660080,0.057197,0.194840,0.098006,0.176233,"POLYGON ((85.3243 27.62037, 85.32434 27.62104,..."
1,8a3c029962effff,19,58,282,2,10,65,12,31,209,...,1300.842448,3.094618,5.152504,0.362281,0.377424,0.049147,0.197054,0.174635,0.155071,"POLYGON ((85.28985 27.70353, 85.28988 27.7042,..."
2,8a3c02890d5ffff,0,0,0,0,0,1,0,1,2,...,1876.812934,11.436138,22.998011,0.094594,0.974060,0.000268,0.015845,0.004104,0.007821,"POLYGON ((85.42076 27.61666, 85.42079 27.61733..."
3,8a3c02882427fff,0,1,3,0,0,1,0,0,1,...,1792.487870,10.656623,23.396290,0.102423,0.957304,0.000114,0.011362,0.002807,0.005519,"POLYGON ((85.45287 27.63726, 85.4529 27.63793,..."
4,8a3c02d4a91ffff,1,1,7,0,0,3,0,0,5,...,1414.921191,8.806582,23.186688,0.283986,0.774673,0.003296,0.028481,0.018736,0.028488,"POLYGON ((85.35036 27.60056, 85.35039 27.60124..."


In [19]:
# Modular


In [21]:
import joblib
import geopandas as gpd
import numpy as np
import pandas as pd
from pathlib import Path

BASE_DIR = Path(r"C:\Users\anils\Desktop\7th SEM\GIS\GeoOptim\Output")

In [28]:
def infer(model_path, scaler_path, data_path, output_path, single=False):
    print("⏳ Loading model pipeline artifacts...")
    scaler = joblib.load(BASE_DIR/scaler_path)
    model = joblib.load(BASE_DIR/model_path)

    hex_gdf = gpd.read_file(data_path)
    # Define the non-feature tracking attributes to isolate your dynamic feature matrix
    metadata_cols = ['h3_id', 'geometry']
    feature_cols = [col for col in hex_gdf.columns if col not in metadata_cols and not col.startswith('prob_')]

    # The target multi-label classes your network yields output arrays for
    clean_label_names = [
        'Restaurant', 'Hotel_Lodging', 'Malls_Department', 'Cafe_Bakery' 
        , 'Fashion_Clothing', 'Convenience_Specialty'
    ]

    X_inference = hex_gdf[feature_cols].copy()
    # Ensure feature counts align with what the scaler expects
    expected_features = scaler.n_features_in_
    actual_features = len(feature_cols)
    if actual_features != expected_features:
        raise ValueError(f"❌ Feature count mismatch! Model expects {expected_features}, but dataset has {actual_features}.")

    # ==============================================================================
    # 4. Run Scaling and Predict Probabilities (FIXED FOR MLP)
    # ==============================================================================
    print("⚖️ Normalizing features using loaded RobustScaler...")
    X_inference_scaled = scaler.transform(X_inference)
    print("🧠 Running Multi-Label MLP model inference engine...")
    # ✨ FIX: For MLP Multi-label, y_prob is a single 2D array of shape (n_samples, n_classes)
    y_prob_matrix = model.predict_proba(X_inference_scaled)

    if single:
        y_prob_matrix = np.column_stack([prob[:, 1] for prob in y_prob_matrix])

    print("📈 Appending class probabilities to dataset...")
    for idx, label_name in enumerate(clean_label_names):
        # Slice the matrix column directly for each target class
        hex_gdf[f'prob_{label_name}'] = y_prob_matrix[:, idx].astype('float32')

    # ==============================================================================
    # 5. Export Updated Geographic Layer File
    # ==============================================================================
    print(f"💾 Saving complete prediction probability field matrix...")
    hex_gdf.to_file(BASE_DIR/output_path, driver="GeoJSON")

    print("\n🎉 Model Inference Run Complete!")
    print(f"Saved completed predictions with {len(clean_label_names)} probability fields")

In [29]:
infer(r"Models/rf_multilabel_model.pkl", r"Models/robust_scaler.pkl", r"C:\Users\anils\Desktop\7th SEM\GIS\GeoOptim\Data\Final\hexagon_ktm_V2.geojson", r"Plots\rf_prediction.geojson", single=True)

⏳ Loading model pipeline artifacts...
⚖️ Normalizing features using loaded RobustScaler...
🧠 Running Multi-Label MLP model inference engine...
📈 Appending class probabilities to dataset...
💾 Saving complete prediction probability field matrix...

🎉 Model Inference Run Complete!
Saved completed predictions with 6 probability fields
